# 🧠 Module 01: The Renaissance — Computational Graphs, Autograd & The Reverse Chain Rule
### *From Linnainmaa (1970) and Rumelhart, Hinton & Williams (1986) to Modern Autodiff*

---

## 📜 Historical Lineage
In the history of Artificial Intelligence, few breakthroughs match the quiet revolution of **Reverse Mode Automatic Differentiation**:
- **Seppo Linnainmaa (1970)**: Published the foundational algorithm for reverse-mode automatic differentiation in his University of Helsinki Master's Thesis: *"The representation of the cumulative rounding error of an algorithm as a Taylor expansion of the local rounding errors"*.
- **Paul Werbos (1974)**: In his Harvard PhD dissertation *"Beyond Regression: New Tools for Prediction and Analysis in the Behavioral Sciences"*, Werbos became the first researcher to propose using reverse AD to train multi-layered artificial neural networks.
- **David Rumelhart, Geoffrey Hinton, & Ronald Williams (1986)**: Published *"Learning representations by back-propagating errors"* in *Nature* (Vol. 323, pp. 533–536), shattering the 17-year First AI Winter and demonstrating that multi-layer perceptrons can discover internal hidden representations to solve non-linear problems.

---

## 💧 The Physical Analogy: The Water Pipe Network with Pressure Gauges
Imagine a branching water pipeline:
1. **The Forward Pass**: Water flows down from input valves ($x, w, b$) through arithmetic mixers ($+, \times$, non-linear pumps). It accumulates into a single terminal reservoir: the **Loss ($L$)**.
2. **The Backward Pass**: We inject a unit surge of hydraulic test pressure at the Loss reservoir ($\frac{\partial L}{\partial L} = 1.0$).
3. **Pressure Superposition**: This pressure surge propagates *backwards* upstream. At each junction, the gauge sensitivity (local partial derivative $\frac{\partial \text{out}}{\partial \text{in}}$) meters how much pressure transmits into upstream pipes.
4. If an upstream pipe feeds into multiple downstream junctions, the backpressures **sum together** (the multivariable Chain Rule).


In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version:   {np.__version__}")


## 1. Why Reverse-Mode AD? The $O(1)$ Miracle
Suppose a neural network has $N = 100,000,000$ parameters and produces a single scalar loss $L \in \mathbb{R}$.
- **Numerical Finite Differences**: Requires $2N$ forward passes ($200,000,000$ forward passes to compute one gradient!).
- **Forward-Mode Autodiff (Dual Numbers)**: Evaluates $\frac{\partial f}{\partial x_i}$ one input variable at a time, requiring $N$ forward passes.
- **Reverse-Mode Autodiff (Linnainmaa 1970 / Rumelhart 1986)**: Computes the full gradient vector $\nabla_\mathbf{w} L$ for **ALL $N$ parameters in a SINGLE backward pass**, with computational cost at most $3\times$ the cost of a single forward evaluation!


In [ ]:
class Value:
    """
    A scalar Value node in a dynamic Directed Acyclic Graph (DAG).
    Tracks forward value, adjoint gradient, ancestors, and backward operator.
    """
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f}, op='{self._op}')"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "Only int/float powers supported"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * (self.data ** (other - 1))) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-other)

    def __truediv__(self, other):
        return self * (other ** -1)

    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,), 'ReLU')
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1.0 / (1.0 + math.exp(-max(-60.0, min(60.0, self.data))))
        out = Value(s, (self,), 'sigmoid')
        def _backward():
            self.grad += (s * (1.0 - s)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1.0 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(max(-60.0, min(60.0, self.data)))
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # Topological sort to order DAG nodes from inputs to outputs
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # Seed the loss gradient: dL/dL = 1.0
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


## 2. Testing the Diamond DAG & Gradient Accumulation
When a variable $x$ is used multiple times, the Chain Rule dictates that gradients along all downstream paths must **sum together**:
$$\frac{\partial L}{\partial x} = \sum_{y \in \text{children}(x)} \frac{\partial L}{\partial y} \frac{\partial y}{\partial x}$$

Let's test $c = x^2 + 3x$ for $x = 5.0$.
$$\frac{\partial c}{\partial x} = 2x + 3 = 2(5) + 3 = 13.0$$


In [ ]:
x = Value(5.0, label='x')
a = x ** 2; a.label = 'x^2'
b = 3.0 * x; b.label = '3x'
c = a + b; c.label = 'c'

c.backward()
print(f"Forward Result: c = {c.data:.2f} (Expected: 40.0)")
print(f"Gradient:   dc/dx = {x.grad:.2f} (Expected: 13.0)")
assert math.isclose(x.grad, 13.0), "Gradient mismatch!


## 3. Side-by-Side Verification with PyTorch 2.x
Let's test an intricate composite expression in our scratch `Value` class and compare it against PyTorch's native C++ autograd engine:
$$z = \frac{(x \cdot w + y)^2}{b + e^x}$$
$$L = \sigma(z) + \text{ReLU}(x \cdot y)$$


In [ ]:
# 1. Scratch Value Engine
vx = Value(2.0, label='x')
vy = Value(-3.0, label='y')
vw = Value(0.5, label='w')
vb = Value(1.2, label='b')

vz = ((vx * vw + vy) ** 2) / (vb + vx.exp())
vL = vz.sigmoid() + (vx * vy).relu()
vL.backward()

# 2. PyTorch Native Tensors
tx = torch.tensor(2.0, dtype=torch.float64, requires_grad=True)
ty = torch.tensor(-3.0, dtype=torch.float64, requires_grad=True)
tw = torch.tensor(0.5, dtype=torch.float64, requires_grad=True)
tb = torch.tensor(1.2, dtype=torch.float64, requires_grad=True)

tz = ((tx * tw + ty) ** 2) / (tb + torch.exp(tx))
tL = torch.sigmoid(tz) + torch.relu(tx * ty)
tL.backward()

print("=" * 60)
print(f"{'Variable':10s} | {'Scratch':14s} | {'PyTorch':14s} | {'Delta':10s}")
print("=" * 60)
for name, v_grad, t_grad in [
    ('dL/dx', vx.grad, tx.grad.item()),
    ('dL/dy', vy.grad, ty.grad.item()),
    ('dL/dw', vw.grad, tw.grad.item()),
    ('dL/db', vb.grad, tb.grad.item()),
]:
    diff = abs(v_grad - t_grad)
    print(f"{name:10s} | {v_grad:+.8f}     | {t_grad:+.8f}     | {diff:.2e}")
print("=" * 60)
print(f"Forward Loss: Scratch = {vL.data:.8f}, PyTorch = {tL.item():.8f}")


## 4. Custom PyTorch Autograd Function (`torch.autograd.Function`)
PyTorch enables developers to write custom differentiable operators by defining a subclass of `torch.autograd.Function` with static `forward` and `backward` methods.
Here we implement GELU (Gaussian Error Linear Unit), the core activation in modern Large Language Models (LLaMA, GPT-4).


In [ ]:
class CustomGELU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        sqrt_2_over_pi = math.sqrt(2.0 / math.pi)
        inner = sqrt_2_over_pi * (x + 0.044715 * torch.pow(x, 3))
        tanh_val = torch.tanh(inner)
        ctx.save_for_backward(x, tanh_val)
        return 0.5 * x * (1.0 + tanh_val)

    @staticmethod
    def backward(ctx, grad_output):
        x, tanh_val = ctx.saved_tensors
        sqrt_2_over_pi = math.sqrt(2.0 / math.pi)
        sech2 = 1.0 - torch.pow(tanh_val, 2)
        d_inner = sqrt_2_over_pi * (1.0 + 3.0 * 0.044715 * torch.pow(x, 2))
        d_gelu = 0.5 * (1.0 + tanh_val) + 0.5 * x * sech2 * d_inner
        return grad_output * d_gelu

test_input = torch.randn(4, dtype=torch.float64, requires_grad=True)
passed = torch.autograd.gradcheck(CustomGELU.apply, test_input, eps=1e-6, atol=1e-4)
print(f"PyTorch gradcheck verified Custom GELU: {passed}")


## 5. Training a Neural Network from Scratch using `Value`
We will now construct a 2-layer Multi-Layer Perceptron (MLP) built entirely on our scalar `Value` autograd engine and train it on the non-linear XOR dataset.


In [ ]:
class Neuron:
    def __init__(self, nin, nonlin='tanh'):
        self.w = [Value(random.uniform(-1.0, 1.0)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin == 'tanh' else act.sigmoid()

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout, nonlin='tanh'):
        self.neurons = [Neuron(nin, nonlin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [
            Layer(sz[i], sz[i+1], nonlin='tanh' if i < len(nouts)-1 else 'sigmoid')
            for i in range(len(nouts))
        ]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

# XOR Dataset
random.seed(42)
X = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
Y = [0.0, 1.0, 1.0, 0.0]

model = MLP(2, [4, 1])
losses = []

# Training Loop
for step in range(120):
    # Forward pass
    total_loss = Value(0.0)
    for xi, yi in zip(X, Y):
        pred = model(xi)
        total_loss += (pred - yi) ** 2
    
    # Backward pass
    for p in model.parameters():
        p.grad = 0.0
    total_loss.backward()

    # Gradient descent update
    lr = 0.5
    for p in model.parameters():
        p.data -= lr * p.grad

    losses.append(total_loss.data)

print(f"Initial Loss: {losses[0]:.4f} -> Final Loss: {losses[-1]:.4f}")
print("\nFinal Predictions on XOR:")
for xi, yi in zip(X, Y):
    pred = model(xi).data
    print(f"  Input: {xi} | Target: {yi} | Prediction: {pred:.4f} (Class: {round(pred)})")

plt.figure(figsize=(8, 4))
plt.plot(losses, color='royalblue', lw=2)
plt.title("Scratch Autograd MLP Training Loss on XOR")
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.grid(True, alpha=0.3)
plt.show()


---
## 🎯 Key Takeaways from Module 01
1. **Reverse AD is Linnainmaa's Gift (1970)**: Forward mode scales with number of parameters; reverse mode scales with number of outputs. Since scalar loss has 1 output, reverse mode makes deep learning computationally feasible.
2. **Topological Order is Mandatory**: In reverse mode, child gradients must be completely evaluated before parents can compute their incoming adjoints.
3. **Gradient Accumulation**: Nodes with multiple downstream consumers must accumulate gradients (`+=`) to account for all computational pathways.
4. **PyTorch's C++ Autograd** operates on the exact same theoretical principles as our `Value` class, vectorized across multidimensional tensors.
